# Chapter 2 - Lab 2: Customer-care agent with working memory

We hold a **six-turn chat** between a customer and a care agent. The conversation history is the
agent's **working memory** (short-term memory).

*Finance version*: a portfolio-rebalancing agent had to remember the current allocation
(70/20/10) across turns to compute new trades.
*Customer-care version*: the care agent must remember the **order, prices, dates, repairs tried and
credits promised** to answer turn 5: *"So exactly how much do I need to pay for the upgrade?"*.
Without memory it cannot do the arithmetic Rs 71,490 - Rs 62,990 - Rs 3,000 = **Rs 5,500**.

In [ ]:
%pip install -q openai-agents

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

In [ ]:
from agents import Agent, Runner, SQLiteSession

# The care agent

The agent has **no tools** here: the policy is written into its instructions. We focus only on memory.

In [ ]:
care_agent = Agent(
    name="NovaMart Care Agent",
    instructions="""You are a senior NovaMart customer-care agent helping a customer by chat.
Your goal is to resolve the customer's problem fairly, quickly and within policy.

Policy you must apply:
- Repeat-failure rule: if the same defect persists after 2 repair attempts within warranty,
  the customer is entitled to a replacement of the same model.
- Upgrades: a customer may upgrade the replacement to a higher model by paying the price difference.
- Goodwill credits: up to Rs 1,000 at agent level, up to Rs 5,000 with supervisor approval.
  Credits can be adjusted against an upgrade payment.
- Large-appliance replacements are delivered and installed within 5-7 working days
  (priority dispatch within 72 hours for Platinum customers when stock is available).

Your role:
- Keep track of the facts the customer gives you (order, prices, dates, what was already tried).
- Do the arithmetic for the customer when money is involved; show your calculation.
- Be empathetic, concise and specific. State assumptions when information is missing.""",
    model="gpt-4o-mini",
)

# Set up working memory

In the OpenAI Agents SDK, working memory is a **session**. `SQLiteSession` stores every message of the
conversation; each `Runner.run(..., session=...)` call reads the history first and appends the new turn.
(Pass a file name, e.g. `SQLiteSession("priya_chat", "care.db")`, to keep it after a restart.)

In [ ]:
session_care = SQLiteSession("priya_refrigerator_case")

In [ ]:
turn_1 = """Hi, I'm Priya Sharma, customer ID C1001. My FrostLine 340L refrigerator (order ORD-77812),
bought for Rs 62,990 and delivered on 18 September, stopped cooling on 9 October. What should I do?"""

turn_2 = """A technician already came on 12 October and replaced a sensor; it failed again two days later.
A second visit on 19 October also failed: they said a relay part is out of stock."""

turn_3 = "So what are my options now?"

turn_4 = """I'll take the replacement, but can I upgrade to the FrostLine 400L model instead? It costs Rs 71,490.
Also, your colleague promised me a Rs 3,000 goodwill credit for the food I had to throw away."""

turn_5 = "So exactly how much do I need to pay for the upgrade?"

turn_6 = "Fine. Please confirm delivery for Thursday and summarise everything we agreed."

In [ ]:
async def show_working_memory(session) -> list[tuple[str, str]]:
    """Print the session (working memory) and return it as (role, text) pairs."""
    items = await session.get_items()
    history = []
    for i, msg in enumerate(items, 1):
        role = msg.get("role")
        content = msg.get("content", "")
        if role == "assistant":
            text = content[0]["text"] if isinstance(content, list) else str(content)
        elif role == "user":
            text = content if isinstance(content, str) else str(content)
        else:
            continue
        history.append((role, text))
        print(f"  {i}. \033[95m{role}\033[0m: {text[:300]}{'...' if len(text) > 300 else ''}")
        print("--" * 40)
    return history

## Turn 1

In [ ]:
output_1 = await Runner.run(care_agent, turn_1, session=session_care)
print(output_1.final_output)

In [ ]:
history = await show_working_memory(session_care)

## Turn 2

In [ ]:
output_2 = await Runner.run(care_agent, turn_2, session=session_care)
print(output_2.final_output)

## Turn 3: the agent must remember *two* failed repairs to apply the repeat-failure rule

In [ ]:
output_3 = await Runner.run(care_agent, turn_3, session=session_care)
print(output_3.final_output)

## Turn 4

In [ ]:
output_4 = await Runner.run(care_agent, turn_4, session=session_care)
print(output_4.final_output)

## Turn 5: the memory test (needs the old price from turn 1 and the credit from turn 4)

In [ ]:
output_5 = await Runner.run(care_agent, turn_5, session=session_care)
print(output_5.final_output)

## Turn 6

In [ ]:
output_6 = await Runner.run(care_agent, turn_6, session=session_care)
print(output_6.final_output)

### Display the final working memory

In [ ]:
history = await show_working_memory(session_care)

### Prove it: the same turn-5 question with no memory

In [ ]:
no_memory = await Runner.run(care_agent, turn_5)   # no session = no history
print(no_memory.final_output)

Without the session the agent cannot know the old price, the new price or the credit. It must ask
again, which is exactly the "please repeat your problem" experience customers hate.

# A first pass at episodic memory

Working memory dies when the chat ends. To remember this case **next week**, we condense the
conversation into a structured **episode** (LangChain prompt -> LLM -> JSON parser). Lab 3 stores and
retrieves these episodes.

In [ ]:
%pip install -q langchain-openai

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [ ]:
episodic_memory_prompt_template = """You are an episodic-memory agent supporting a principal customer-care agent.
Your job is to create a concise episodic memory of a support session:
* Summarise the conversation (conversation_summary),
* Record the case trajectory: how the issue, the options offered and the money involved evolved, with dates (case_trajectory),
* Identify what worked (what_worked) and what did not (what_didnt_work),
* Extract key points to remember next time this customer contacts us (key_points).

You receive the full conversation between the customer and the principal agent.

OUTPUT FORMAT - return only valid JSON in this exact schema:
{{
  "conversation_summary": string,   // 3-5 sentences: customer, product, problem, resolution, money
  "case_trajectory": string,        // ordered steps with dates and amounts
  "what_worked": string,            // e.g. clear policy explanation, correct arithmetic, empathy
  "what_didnt_work": string,        // e.g. repeated failed repairs, slow escalation, unclear commitments
  "key_points": ["string"]          // facts and commitments to remember for future contacts
}}

FEW-SHOT EXAMPLES
GOOD conversation_summary:
"Platinum customer reported a washing machine leaking after one month. Two repairs failed, so the agent
applied the repeat-failure rule and approved a replacement. The customer upgraded to the 8 kg model and paid
Rs 4,200 after a Rs 1,000 goodwill credit. Delivery confirmed for 14 March."
BAD conversation_summary:
"The customer had a problem. The agent helped. It was resolved." (no product, no numbers, no commitments)

GOOD what_didnt_work:
"The customer had to explain the history three times across channels; the second technician visit
was booked without the spare part, wasting a visit."
BAD what_didnt_work:
"Nothing went wrong." (unrealistic; no operational insight)

General rules:
- Be concise and factual. Do not invent facts, amounts or dates that are not in the conversation.
- Output only valid JSON and nothing else.

Here is the conversation between the customer and the principal care agent:
{conversation}
"""

In [ ]:
episodic_memory_prompt = ChatPromptTemplate.from_template(episodic_memory_prompt_template)
generate_episodic_memory = episodic_memory_prompt | llm | JsonOutputParser()

conversation_history_string = "\n\n".join(f'{role}: """{text}"""' for role, text in history)
episode = generate_episodic_memory.invoke({"conversation": conversation_history_string})
episode

## Discussion
1. Where does working memory live in your contact centre today (CRM notes, agent's head, ticket thread)?
2. What should **never** be stored in memory (card numbers, OTPs, health data)? How would you enforce it?